# Superstore Data Pipeline & Advanced SQL Analysis
**Environment:** Python (Pandas, SQLite3) | **Core Concepts:** Data Normalization, CTEs, Window Functions, Subqueries

## Setup and In-Memory Database Initialization
This initialization block establishes a local data pipeline. 
* **Pandas** is utilized to read the raw CSV file. The `windows-1252` encoding is explicitly set to handle specific character formatting common in the Superstore dataset.
* **SQLite3** is used to create an in-memory database (`:memory:`). This allows us to execute standard SQL syntax directly against our dataframe without requiring external server configuration or leaving residual `.db` files on the local machine.

In [1]:
import pandas as pd
import sqlite3

# 1. Load the raw dataset
df = pd.read_csv('Sample - Superstore.csv', encoding='windows-1252')

# 2. Initialize in-memory SQLite database
conn = sqlite3.connect(':memory:')

# 3. Write dataframe to SQL database as the foundational 'raw' table
df.to_sql('superstore_raw', conn, if_exists='replace', index=False)

print("Database connected. Raw Superstore data loaded successfully.")

Database connected. Raw Superstore data loaded successfully.


## Step 1: Data Normalization
The raw dataset is a flat file containing redundant entity information. To adhere to relational database principles, we normalize `superstore_raw` into three distinct dimensional and fact tables:
1. **`customers`**: Unique customer demographic data.
2. **`products`**: Unique product hierarchy data.
3. **`orders`**: Transactional fact data linking customers and products.

> **Methodology:** We utilize `SELECT DISTINCT` to extract unique records and eliminate duplicates during table creation.

In [2]:
setup_queries = [
    """
    CREATE TABLE customers AS 
    SELECT DISTINCT "Customer ID", "Customer Name", "Segment", "Country", "City", "State", "Postal Code", "Region" 
    FROM superstore_raw;
    """,
    """
    CREATE TABLE products AS 
    SELECT DISTINCT "Product ID", "Category", "Sub-Category", "Product Name" 
    FROM superstore_raw;
    """,
    """
    CREATE TABLE orders AS 
    SELECT DISTINCT "Row ID", "Order ID", "Order Date", "Ship Date", "Ship Mode", "Customer ID", "Product ID", "Sales", "Quantity", "Discount", "Profit" 
    FROM superstore_raw;
    """
]

cursor = conn.cursor()
for query in setup_queries:
    cursor.execute(query)
conn.commit()

print("Relational schema built: `customers`, `products`, and `orders` tables created.")

Relational schema built: `customers`, `products`, and `orders` tables created.


## Step 2: Advanced SQL Operations
This section executes the assignment's core requirements, demonstrating proficiency with complex SQL clauses.

**Techniques Applied:**
* **Subqueries:** Used in the `WHERE` clause to filter dynamically (e.g., finding sales above the calculated average).
* **CTEs (Common Table Expressions):** Utilized (`WITH` clause) to create temporary result sets for customer aggregations, improving query readability.
* **Window Functions:** * `RANK() OVER()`: Used to rank aggregated customer sales without collapsing rows.
  * `ROW_NUMBER() OVER(PARTITION BY ...)`: Applied to sequence chronological orders uniquely per customer.

In [3]:
# 1. Orders > Average Sales (Subquery)
q1 = 'SELECT "Order ID", "Sales" FROM orders WHERE "Sales" > (SELECT AVG("Sales") FROM orders);'
print("\n--- 1. Orders Above Average Sales ---")
display(pd.read_sql_query(q1, conn).head(3))

# 2. Highest sales order per customer (Correlated Subquery)
q2 = """
SELECT o1."Customer ID", o1."Order ID", o1."Sales" 
FROM orders o1
WHERE o1."Sales" = (SELECT MAX(o2."Sales") FROM orders o2 WHERE o1."Customer ID" = o2."Customer ID");
"""
print("\n--- 2. Highest Sales Order Per Customer ---")
display(pd.read_sql_query(q2, conn).head(3))

# 3. Total sales per customer (CTE)
q3 = 'WITH CustomerSales AS (SELECT "Customer ID", SUM("Sales") as TotalSales FROM orders GROUP BY "Customer ID") SELECT * FROM CustomerSales;'
print("\n--- 3. Total Sales Per Customer ---")
display(pd.read_sql_query(q3, conn).head(3))

# 4. Customers with above-average total sales (CTE + Subquery)
q4 = """
WITH CustomerSales AS (SELECT "Customer ID", SUM("Sales") as TotalSales FROM orders GROUP BY "Customer ID")
SELECT * FROM CustomerSales WHERE TotalSales > (SELECT AVG(TotalSales) FROM CustomerSales);
"""
print("\n--- 4. Customers With Above-Average Total Sales ---")
display(pd.read_sql_query(q4, conn).head(3))

# 5. Rank customers by total sales (Window Function)
q5 = 'SELECT "Customer ID", SUM("Sales") as TotalSales, RANK() OVER(ORDER BY SUM("Sales") DESC) as SalesRank FROM orders GROUP BY "Customer ID";'
print("\n--- 5. Customer Sales Ranking ---")
display(pd.read_sql_query(q5, conn).head(3))

# 6. Assign row numbers to orders within a customer (Window Function + PARTITION BY)
q6 = 'SELECT "Customer ID", "Order ID", "Order Date", ROW_NUMBER() OVER(PARTITION BY "Customer ID" ORDER BY "Order Date" ASC) as OrderSeq FROM orders;'
print("\n--- 6. Chronological Order Sequence Per Customer ---")
display(pd.read_sql_query(q6, conn).head(5))

# 7. Top 3 customers by total sales (CTE + Window Function)
q7 = """
WITH Ranked AS (SELECT "Customer ID", SUM("Sales") as TotalSales, RANK() OVER(ORDER BY SUM("Sales") DESC) as SalesRank FROM orders GROUP BY "Customer ID")
SELECT * FROM Ranked WHERE SalesRank <= 3;
"""
print("\n--- 7. Top 3 Customers Overall ---")
display(pd.read_sql_query(q7, conn))


--- 1. Orders Above Average Sales ---


,Order ID,Sales
0,CA-2016-152156,261.9600
1,CA-2016-152156,731.9400
2,US-2015-108966,957.5775



--- 2. Highest Sales Order Per Customer ---


,Customer ID,Order ID,Sales
0,CG-12520,CA-2016-152156,731.9400
1,SO-20335,US-2015-108966,957.5775
2,BH-11710,CA-2014-115812,1706.1840



--- 3. Total Sales Per Customer ---


...

,Customer ID,TotalSales,SalesRank
0,SM-20320,25043.050,1
1,TC-20980,19052.218,2
2,RB-19360,15117.339,3



--- 6. Chronological Order Sequence Per Customer ---


,Customer ID,Order ID,Order Date,OrderSeq
0,AA-10315,CA-2015-121391,10/4/2015,1
1,AA-10315,CA-2016-103982,3/3/2016,2
2,AA-10315,CA-2016-103982,3/3/2016,3
3,AA-10315,CA-2016-103982,3/3/2016,4
4,AA-10315,CA-2016-103982,3/3/2016,5



--- 7. Top 3 Customers Overall ---


,Customer ID,TotalSales,SalesRank
0,SM-20320,25043.050,1
1,TC-20980,19052.218,2
2,RB-19360,15117.339,3


## Step 3: Final Synthesized Query
This query combines three distinct SQL concepts into a single execution plan to output a clean, readable leaderboard.
1. **`JOIN`**: Maps the `orders` fact table to the `customers` dimension table to retrieve the human-readable `Customer Name`.
2. **`CTE`**: Aggregates the total sales per customer name in a temporary block.
3. **`Window Function`**: Applies `RANK()` over the aggregated data from the CTE to establish a definitive hierarchy.

In [4]:
final_query = """
WITH CustomerSales AS (
    SELECT c."Customer Name", SUM(o."Sales") as TotalSales
    FROM orders o
    JOIN customers c ON o."Customer ID" = c."Customer ID"
    GROUP BY c."Customer Name"
)
SELECT "Customer Name", TotalSales, RANK() OVER(ORDER BY TotalSales DESC) as SalesRank
FROM CustomerSales;
"""
print("--- Final Combined Query Output ---")
display(pd.read_sql_query(final_query, conn).head(10))

--- Final Combined Query Output ---


,Customer Name,TotalSales,SalesRank
0,Ken Lonsdale,155927.519,1
1,Sanjit Engle,134303.818,2
2,Clay Ludtke,130566.552,3
3,Adrian Barton,130262.139,4
4,Sanjit Chand,127281.006,5
5,Sean Miller,125215.250,6
6,Edward Hooks,123730.560,7
7,Greg Tran,118201.200,8
8,Seth Vernon,114709.500,9
9,John Lee,107799.153,10


## Mini Project: Customer Sales Insights
Extracting actionable business intelligence from the normalized tables. 

* **Top/Bottom Performers:** Utilizing `ORDER BY` coupled with `LIMIT` to isolate outliers at both ends of the revenue spectrum.
* **Customer Retention:** Using the `HAVING` clause to filter grouped aggregates, specifically identifying churned or single-interaction users (`COUNT(DISTINCT Order ID) = 1`).

In [5]:
# 1. Top 5 customers
print("--- 1. Top 5 Customers ---")
display(pd.read_sql_query('SELECT c."Customer Name", SUM(o."Sales") as TotalSales FROM orders o JOIN customers c ON o."Customer ID" = c."Customer ID" GROUP BY c."Customer Name" ORDER BY TotalSales DESC LIMIT 5;', conn))

# 2. Bottom 5 customers
print("\n--- 2. Bottom 5 Customers ---")
display(pd.read_sql_query('SELECT c."Customer Name", SUM(o."Sales") as TotalSales FROM orders o JOIN customers c ON o."Customer ID" = c."Customer ID" GROUP BY c."Customer Name" ORDER BY TotalSales ASC LIMIT 5;', conn))

# 3. One-time buyers (Single Order)
print("\n--- 3. Single-Order Customers ---")
display(pd.read_sql_query('SELECT c."Customer Name", COUNT(DISTINCT o."Order ID") as OrderCount FROM orders o JOIN customers c ON o."Customer ID" = c."Customer ID" GROUP BY c."Customer Name" HAVING OrderCount = 1;', conn).head(5))

# 4. Above-average sales
print("\n--- 4. Above Average Customer Base ---")
display(pd.read_sql_query('WITH CTotals AS (SELECT c."Customer Name", SUM(o."Sales") as T FROM orders o JOIN customers c ON o."Customer ID" = c."Customer ID" GROUP BY c."Customer Name") SELECT "Customer Name", T as TotalSales FROM CTotals WHERE T > (SELECT AVG(T) FROM CTotals);', conn).head(5))

# 5. Highest order value per customer
print("\n--- 5. Max Order Value Per Customer ---")
display(pd.read_sql_query('SELECT c."Customer Name", MAX(o."Sales") as MaxOrderValue FROM orders o JOIN customers c ON o."Customer ID" = c."Customer ID" GROUP BY c."Customer Name";', conn).head(5))

# Close the database connection to free up memory
conn.close()

--- 1. Top 5 Customers ---


,Customer Name,TotalSales
0,Ken Lonsdale,155927.519
1,Sanjit Engle,134303.818
2,Clay Ludtke,130566.552
3,Adrian Barton,130262.139
4,Sanjit Chand,127281.006



--- 2. Bottom 5 Customers ---


,Customer Name,TotalSales
0,Lela Donovan,5.304
1,Thais Sissman,9.666
2,Carl Jackson,16.520
3,Mitch Gastineau,16.739
4,Roy Skaria,44.656



--- 3. Single-Order Customers ---


,Customer Name,OrderCount
0,Anemone Ratner,1
1,Anthony O'Donnell,1
2,Carl Jackson,1
3,Jenna Caffey,1
4,Jocasta Rupert,1



--- 4. Above Average Customer Base ---


,Customer Name,TotalSales
0,Aaron Smayling,21354.844
1,Adam Bellavance,62044.960
2,Adam Hart,32503.370
3,Adam Shillingsburg,26042.480
4,Adrian Barton,130262.139



--- 5. Max Order Value Per Customer ---


,Customer Name,MaxOrderValue
0,Aaron Bergman,341.960
1,Aaron Hawkins,668.160
2,Aaron Smayling,1439.982
3,Adam Bellavance,4355.168
4,Adam Hart,841.568
